# CredibleX Growth Intelligence Engine
## 03 · AI Intelligence Layer

This notebook is where the project becomes differentiated: instead of using
an LLM for generic text generation, it is used to **explain calculated
analytical results** in business language.

For each selected SME, the AI generates an **AI Financing Brief**:
`Business Profile → Financing Opportunity → Recommended Product → Key
Drivers → Potential Concern → Suggested Next Action`.

For each partner, an **AI Partner Brief** with an equivalent structure.
Finally, an **AI Executive Summary** synthesises portfolio-level insights.

### How this works
Prompts are built strictly from each row's own analytical fields (see
`../ai/prompts.md`) — the model is asked to *explain*, never to invent,
numbers. If an `ANTHROPIC_API_KEY` is set in the environment, this notebook
calls the live Claude API; otherwise it falls back to a deterministic
template generator with the same structure, so the notebook always runs
end-to-end without credentials.

In [1]:
import pandas as pd
import json
import os

pd.set_option('display.max_colwidth', 200)

sme = pd.read_csv('../data/sme_scored.csv')
partner = pd.read_csv('../data/partner_scored.csv')
print(f"Live LLM available: {bool(os.environ.get('ANTHROPIC_API_KEY') or os.environ.get('OPENAI_API_KEY'))}")
print(sme.shape, partner.shape)

Live LLM available: False
(1000, 19) (50, 11)


### Prompt template (SME brief)
See `../ai/prompts.md` for the full prompt library.

In [2]:
def build_sme_prompt(row):
    return f'''You are a financing analyst at CredibleX writing an internal brief.
Use ONLY the facts below. Do not invent numbers. Keep it concise and business-toned.

SME_ID: {row['SME_ID']}
Industry: {row['Industry']}
Emirate: {row['Emirate']}
Monthly_Revenue (AED): {row['Monthly_Revenue']}
Revenue_Growth: {row['Revenue_Growth']}
Receivable_Days: {row['Receivable_Days']}
Cash_Buffer (days): {row['Cash_Buffer']}
Financing_Requirement (AED): {row['Financing_Requirement']}
Opportunity_Score: {row['Opportunity_Score']} ({row['Opportunity_Segment']})
Recommended_Product: {row['Recommended_Product']}
Early_Warning_Status: {row['Early_Warning_Status']}

Return a JSON object with keys: business_profile, financing_opportunity,
recommended_product, key_drivers, potential_concern, suggested_next_action.'''

print(build_sme_prompt(sme.iloc[0]))

You are a financing analyst at CredibleX writing an internal brief.
Use ONLY the facts below. Do not invent numbers. Keep it concise and business-toned.

SME_ID: SME0001
Industry: Professional Services
Emirate: Dubai
Monthly_Revenue (AED): 58064
Revenue_Growth: 0.007
Receivable_Days: 106
Cash_Buffer (days): 6
Financing_Requirement (AED): 99319
Opportunity_Score: 50.3 (Emerging Opportunity)
Recommended_Product: Receivables Financing
Early_Warning_Status: Attention Required

Return a JSON object with keys: business_profile, financing_opportunity,
recommended_product, key_drivers, potential_concern, suggested_next_action.


### Template-based generator (fallback / offline-safe path)
Mirrors the exact structure an LLM response would follow.

In [3]:
def fmt_pct(x): return f"{x*100:.1f}%"
def fmt_aed(x): return f"AED {x:,.0f}"

def generate_sme_brief(row):
    growth_desc = "growing" if row['Revenue_Growth'] > 0.03 else ("contracting" if row['Revenue_Growth'] < -0.03 else "broadly flat")
    volatility_desc = "stable" if row['Revenue_Volatility'] < 0.3 else "variable"

    business_profile = (
        f"{row['SME_ID']} is a {row['Business_Age']}-year-old {row['Industry']} business based in "
        f"{row['Emirate']}, generating approximately {fmt_aed(row['Monthly_Revenue'])} in monthly "
        f"revenue with {volatility_desc} revenue patterns and revenue that is currently {growth_desc} "
        f"({fmt_pct(row['Revenue_Growth'])} growth).")

    financing_opportunity = (
        f"The business scores {row['Opportunity_Score']}/100 on the Financing Opportunity Score, "
        f"placing it in the \'{row['Opportunity_Segment']}\' segment, with an estimated illustrative "
        f"financing requirement of {fmt_aed(row['Financing_Requirement'])}.")

    recommended_product = f"{row['Recommended_Product']}. {row['Recommendation_Reason']}"

    drivers = []
    if row['Receivable_Days'] > 45: drivers.append(f"elevated receivable days ({row['Receivable_Days']} days)")
    if row['Revenue_Growth'] > 0.1: drivers.append(f"strong revenue growth ({fmt_pct(row['Revenue_Growth'])})")
    if row['Transaction_Volume'] > 150: drivers.append(f"healthy transaction activity ({row['Transaction_Volume']} txns/mo)")
    if row['Cash_Buffer'] < 20: drivers.append(f"thin cash buffer ({row['Cash_Buffer']} days)")
    if not drivers: drivers.append("a moderate, well-balanced financial profile")
    key_drivers = "; ".join(drivers).capitalize() + "."

    if row['Early_Warning_Status'] == 'Attention Required':
        potential_concern = "Multiple early-warning indicators are flagged \u2014 this SME warrants closer review before any financing offer is extended."
    elif row['Early_Warning_Status'] == 'Monitor':
        potential_concern = "One or two early-warning signals are present; not disqualifying, but worth monitoring."
    else:
        potential_concern = "No material early-warning signals are present at this time."

    suggested_next_action = (
        f"Prioritise outreach with a tailored {row['Recommended_Product']} offer" +
        (", paired with a light-touch financial review given the flagged concerns." if row['Early_Warning_Status'] != 'Stable' else " and fast-track given the clean risk profile."))

    return {
        "business_profile": business_profile, "financing_opportunity": financing_opportunity,
        "recommended_product": recommended_product, "key_drivers": key_drivers,
        "potential_concern": potential_concern, "suggested_next_action": suggested_next_action,
    }

sample_brief = generate_sme_brief(sme.iloc[0])
print(json.dumps(sample_brief, indent=2))

{
  "business_profile": "SME0001 is a 2.0-year-old Professional Services business based in Dubai, generating approximately AED 58,064 in monthly revenue with stable revenue patterns and revenue that is currently broadly flat (0.7% growth).",
  "financing_opportunity": "The business scores 50.3/100 on the Financing Opportunity Score, placing it in the 'Emerging Opportunity' segment, with an estimated illustrative financing requirement of AED 99,319.",
  "recommended_product": "Receivables Financing. Receivable days of 106 exceed payable days of 34, suggesting delayed customer payments are creating working-capital pressure that invoice/receivables financing could bridge.",
  "key_drivers": "Elevated receivable days (106 days); thin cash buffer (6 days).",
  "potential_concern": "Multiple early-warning indicators are flagged \u2014 this SME warrants closer review before any financing offer is extended.",
  "suggested_next_action": "Prioritise outreach with a tailored Receivables Financing

### Optional live-LLM call
Activates automatically if an API key is present in the environment; otherwise this cell is skipped.

In [4]:
import urllib.request

def call_claude(prompt):
    req = urllib.request.Request(
        "https://api.anthropic.com/v1/messages",
        data=json.dumps({
            "model": "claude-sonnet-4-6", "max_tokens": 600,
            "messages": [{"role": "user", "content": prompt}],
        }).encode(),
        headers={"Content-Type": "application/json",
                 "x-api-key": os.environ.get("ANTHROPIC_API_KEY", ""),
                 "anthropic-version": "2023-06-01"},
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.loads(resp.read())
    text = "".join(b["text"] for b in data["content"] if b["type"] == "text")
    text = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(text)

USE_LIVE_LLM = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"USE_LIVE_LLM = {USE_LIVE_LLM}")
if USE_LIVE_LLM:
    print(json.dumps(call_claude(build_sme_prompt(sme.iloc[0])), indent=2))
else:
    print("No API key found in this environment — using the template generator above for all briefs.")

USE_LIVE_LLM = False
No API key found in this environment — using the template generator above for all briefs.


## Generate AI Financing Briefs for the Top SMEs
Generating briefs for the top 20 High-Opportunity SMEs (illustrative — in production this would run for the full outreach list).

In [5]:
def generate_partner_brief(row):
    partner_profile = (
        f"{row['Partner_ID']} is a {row['Partner_Type']} with a primary focus on the "
        f"{row['Industry_Focus']} sector, reaching an estimated {row['SME_Reach']:,} SMEs and "
        f"processing roughly {fmt_aed(row['Transaction_Volume'])} in transaction volume per month.")
    sme_exposure = (
        f"With {row['SME_Reach']:,} SMEs in its network and a digital maturity score of "
        f"{row['Digital_Maturity']}/100, this partner offers "
        f"{'substantial' if row['SME_Reach'] > 2000 else 'moderate'} reach into CredibleX's target segment.")
    financing_opportunity = (
        f"Financing relevance is scored at {row['Financing_Relevance']}/100, and the partner's "
        f"overall Partner Opportunity Score is {row['Partner_Score']}/100 (\'{row['Partner_Segment']}\').")
    potential_fit = (
        "Strong candidate for an embedded-finance integration at the point of transaction or invoicing."
        if row['Partner_Segment'] == 'Priority' else
        "Reasonable candidate worth developing further as digital integration matures."
        if row['Partner_Segment'] == 'Emerging' else
        "Lower near-term fit; better suited to periodic monitoring than active pursuit.")
    why_it_matters = (
        f"Growth rate of {fmt_pct(row['Growth_Rate'])} and {row['Integration_Complexity'].lower()} "
        f"integration complexity mean this partnership could scale "
        f"{'quickly' if row['Growth_Rate'] > 0.15 and row['Integration_Complexity']=='Low' else 'steadily'} if pursued.")
    suggested_bd_angle = (
        f"Approach with an embedded-financing pilot tailored to {row['Industry_Focus']} SMEs in the "
        f"partner's network, emphasising {'quick technical integration' if row['Integration_Complexity']=='Low' else 'a phased integration roadmap'}.")
    return {
        "partner_profile": partner_profile, "sme_exposure": sme_exposure,
        "financing_opportunity": financing_opportunity, "potential_fit": potential_fit,
        "why_it_matters": why_it_matters, "suggested_bd_angle": suggested_bd_angle,
    }

top_smes = sme[sme['Opportunity_Segment']=='High Opportunity'].nlargest(20, 'Opportunity_Score')
sme_briefs = {row['SME_ID']: generate_sme_brief(row) for _, row in top_smes.iterrows()}

top_partners = partner.nlargest(20, 'Partner_Score')
partner_briefs = {row['Partner_ID']: generate_partner_brief(row) for _, row in top_partners.iterrows()}

print(f"Generated {len(sme_briefs)} SME briefs and {len(partner_briefs)} partner briefs")

Generated 20 SME briefs and 20 partner briefs


In [6]:
# Preview one of each
print("=== SAMPLE SME BRIEF ===")
sample_id = list(sme_briefs.keys())[0]
print(sample_id)
print(json.dumps(sme_briefs[sample_id], indent=2))

print("\n=== SAMPLE PARTNER BRIEF ===")
sample_pid = list(partner_briefs.keys())[0]
print(sample_pid)
print(json.dumps(partner_briefs[sample_pid], indent=2))

=== SAMPLE SME BRIEF ===
SME0046
{
  "business_profile": "SME0046 is a 2.1-year-old Professional Services business based in Dubai, generating approximately AED 424,660 in monthly revenue with variable revenue patterns and revenue that is currently growing (44.8% growth).",
  "financing_opportunity": "The business scores 77.8/100 on the Financing Opportunity Score, placing it in the 'High Opportunity' segment, with an estimated illustrative financing requirement of AED 893,585.",
  "recommended_product": "Receivables Financing. Receivable days of 133 exceed payable days of 15, suggesting delayed customer payments are creating working-capital pressure that invoice/receivables financing could bridge.",
  "key_drivers": "Elevated receivable days (133 days); strong revenue growth (44.8%); healthy transaction activity (814 txns/mo).",
  "potential_concern": "One or two early-warning signals are present; not disqualifying, but worth monitoring.",
  "suggested_next_action": "Prioritise outreac

## AI Executive Summary
Synthesises the four headline insights shown on the dashboard's AI Insights page.

In [7]:
def generate_executive_summary(sme_df, partner_df):
    top_industry = sme_df[sme_df['Opportunity_Segment']=='High Opportunity']['Industry'].value_counts().idxmax()
    top_product = sme_df['Recommended_Product'].value_counts().idxmax()
    pct_high = (sme_df['Opportunity_Segment']=='High Opportunity').mean()*100
    pct_attention = (sme_df['Early_Warning_Status']=='Attention Required').mean()*100
    n_priority = (partner_df['Partner_Segment']=='Priority').sum()
    top_partner_type = partner_df[partner_df['Partner_Segment']=='Priority']['Partner_Type'].value_counts().idxmax()

    return {
        "portfolio_insight": (
            f"{pct_high:.1f}% of analysed SMEs fall into the High-Opportunity segment, with "
            f"{top_industry} the most represented industry among them \u2014 indicating where "
            "embedded-finance demand is currently concentrated."),
        "product_insight": (
            f"{top_product} is the most frequently recommended product across the portfolio, "
            "suggesting working-capital timing mismatches are the dominant financing need among "
            "the SMEs analysed."),
        "partner_insight": (
            f"{n_priority} partners are classified as Priority, led by {top_partner_type} players "
            "\u2014 these represent the strongest near-term candidates for embedded-finance distribution."),
        "early_warning_insight": (
            f"{pct_attention:.1f}% of SMEs are flagged as \'Attention Required\' on the early-warning "
            "framework and should be reviewed before financing outreach, even where their opportunity "
            "score looks attractive."),
    }

exec_summary = generate_executive_summary(sme, partner)
print(json.dumps(exec_summary, indent=2))

{
  "portfolio_insight": "5.9% of analysed SMEs fall into the High-Opportunity segment, with F&B the most represented industry among them \u2014 indicating where embedded-finance demand is currently concentrated.",
  "product_insight": "Revenue-Based Financing is the most frequently recommended product across the portfolio, suggesting working-capital timing mismatches are the dominant financing need among the SMEs analysed.",
  "partner_insight": "15 partners are classified as Priority, led by B2B Marketplace players \u2014 these represent the strongest near-term candidates for embedded-finance distribution.",
  "early_warning_insight": "11.7% of SMEs are flagged as 'Attention Required' on the early-warning framework and should be reviewed before financing outreach, even where their opportunity score looks attractive."
}


## Save AI outputs for the dashboard

In [8]:
output = {
    "sme_briefs": sme_briefs,
    "partner_briefs": partner_briefs,
    "executive_summary": exec_summary,
}
with open('../dashboard/ai_briefs.json', 'w') as f:
    json.dump(output, f, indent=2)
print("Saved ../dashboard/ai_briefs.json")
print(f"  - {len(sme_briefs)} SME briefs")
print(f"  - {len(partner_briefs)} partner briefs")
print("  - 1 executive summary")

Saved ../dashboard/ai_briefs.json
  - 20 SME briefs
  - 20 partner briefs
  - 1 executive summary


## Summary

This notebook demonstrates the intended pattern for the AI layer: **the model
explains results it is given, it does not generate them.** All scores,
segments, and product recommendations come from the deterministic analysis
in `opportunity_engine.ipynb` — the AI layer's job is purely to translate
those numbers into a business-readable narrative for SME outreach and
partner BD conversations.

**Next:** these briefs power the interactive dashboard (`../dashboard/index.html`).